# Research Notebook
## Rosebud John
## Date: 9 February 2026

# 1. Experience
## 1.1 Describe at least one research activity you worked on this week. 

- Coded rotation curves for four snapshots, 105,111,139 and 153, to study the mass infall and concentration over time of the LMC.
- Attended weekly group meeting to give updates and discuss further steps.
- Started brainstorming ideas and concepts for tracking particles.

## 1.2 Motivation:
Plotting velocity and angular momentum rotation curves give us information about how the mass is moving over time, as the velocity depends on the mass. A position - position plot for different snapshots tell us about the movement of particles with respect to time and also added mass, and gives us an idea about how to track particles.

# 2: What? (What happened?)
## 2.1 Describe what happened during your activities for the week.

I wrote a code to plot velocity and angular momentum rotation curves for 4 different snapshots of interest. The plots are based on the work of Calore et al. in the paper _Simulated Milky Way analogues: implications for dark matter indirect searches_ (Section 3.1). 
The equations to calculate ω(R) and v(R) are,
$$ ω(R) = \frac{v(R)}{R} $$
where ω(R) is the angular momentum and v(R) is the circular velocity, given by
$$ v(R) = \frac{\sqrt{GM(<R)}}{R} $$ 
M(<R) is the mass enclosed in R. R is the galactocentric distance give by the distance formula,
$$ R= \sqrt{x^2+y^2+z^2}$$

The code to plot V(R) vs R and ω(R) vs R is below.

```
def get_rotation_data(coords, mass_star, mass_gas, mass_dm=3e5, G=4.30091e-6, step=1000):
    coord = coords[::step]


    # calculating R
    step = 1000
    
    x=coord[:, 0]
    y=coord[:, 1]
    z=coord[:, 2]
    
    R = [] 
    for i in range (0, len(x)):
        r = (np.sqrt(x[i]**2+y[i]**2+z[i]**2))
        R.append(r)
    R = np.array(R)

    # sorting R
    idx = np.argsort(R)
    R = R[idx]

    # calculating M(<R)
    total_mass = mass_star[idx] + mass_gas[idx] + mass_dm

    M_lessR = np.cumsum(total_mass)

    # calculating v(R) and ω(R)
    v_R = np.sqrt(G * M_lessR / R)
    ω_R = v_R/R

    return R, ω_R, v_R

def rotation_from_snapshot(snaptar):
    coords, vel_dm, subflag_dm, subhalo_pos, subhalo_vel, subgroupnum, coord_star, vel_star, subflag_star, mass_star, coord_gas, vel_gas, subflag_gas, mass_gas, coord_bh, vel_bh, subflag_bh, mass_bh = read_in_lmc_data(snaptar)
    return get_rotation_data(coords, mass_star, mass_gas)


for snap in snapshots:
    R, omega_R, v_R = rotation_from_snapshot(snap)
    plt.plot(R, v_R, label=f"snap {snap}")
plt.xlabel("R (kpc)")
plt.ylabel("v(R) (km/s)")
plt.xlim(0, 200)
plt.grid()
plt.legend()
plt.show()


for snap in snapshots:
    R, omega_R, v_R = rotation_from_snapshot(snap)
    plt.plot(R, omega_R, linestyle="--", label=f"snap {snap}")
plt.xlabel("R (kpc)")
plt.ylabel("ω(R) (km/s/kpc)")
plt.xlim(0, 200)
plt.grid()
plt.legend()
plt.show()
```

We also worked on plotting all of the subhalo position data and locating the LMC and MW, just to get a visial idea of the position dispersal. Additionally, we worked on writing a basic code to track every nth DM particle, get the particle IDs and plot each X coordinate against its Y coordiant. This will form a basis of the tracking code we will need to study the LMC particle's movements. NB: Even though we call the variables _lmc, those are infact not LMC only particles. It is just a placeholder name of what we expect the code to ideally do.
The code we wrote is as below:
```
path = '/cosma7/data/dp004/azadehf/Auriga/rerun_halo25/halo_25/output'

snapshots = [105,111,139,153]
step = 2000

#snapshot 105 is the reference 
snap_ref = snapshots[0]

(coord_ref, vel_ref, subflag_ref, subhalo_pos_ref, subhalo_vel_ref, subgroupnum_ref, *_) = read_in_lmc_data(snap_ref)
_, ids_ref   = readsnap_auriga(snap_ref, 'ParticleIDs', 1, indir=path)

#plot subhalos
plt.figure()
plt.scatter(subhalo_pos_ref[:,0], subhalo_pos_ref[:,1], alpha=0.2, label='Subhalos')
plt.scatter(subhalo_pos_ref[0, 0], subhalo_pos_ref[0, 1], marker='*', label='MW')
plt.scatter(subhalo_pos_ref[1, 0], subhalo_pos_ref[1, 1], marker='o', label='LMC')
plt.legend()

#sort id
sort_idx = np.argsort(ids_ref)
ids_ref = ids_ref[sort_idx]

lmc_idx = 1
lmc_mask_ref = (subflag_ref == lmc_idx) # select LMC particles at reference snapshot
coord_lmc_ref = coord_ref[lmc_mask_ref]
print('length coord lmc: ', len(coord_lmc_ref))
ids_lmc_ref   = ids_ref[lmc_mask_ref]

coord_lmc_ref = coord_lmc_ref[::step]
print('length coord lmc sample: ', len(coord_lmc_ref))
ids_lmc_ref   = ids_lmc_ref[::step]

print('LMC center: ',plt.scatter(subhalo_pos[lmc_idx,0], subhalo_pos[lmc_idx,1],marker='o', s=10, label='LMC center') )
print("x range:", coord_lmc_ref[:,0].min(), coord_lmc_ref[:,0].max())
print("y range:", coord_lmc_ref[:,1].min(), coord_lmc_ref[:,1].max())
#print("Number of traced particles:", ids_lmc_ref_sample.size)

tracked_coords = {}  # dictionary: snapshot -> coordinates
for snap in snapshots:
    print(f"snapshot {snap}")
    (coord_dm, vel, subflag, subhalo_pos, subhalo_vel, subgroupnum, *_) = read_in_lmc_data(snap)
    _, ids_dm   = readsnap_auriga(snap, 'ParticleIDs', 1, indir=path)
    ids_dm = ids_dm[np.argsort(ids_dm)]
    
    lmc_center = np.array([subhalo_pos[lmc_idx,0], subhalo_pos[lmc_idx,1]])
    print('lmc center:', lmc_center)
    sort_idx = np.argsort(ids_dm)
    ids_dm = ids_dm[sort_idx]
    coord_dm = coord_dm[sort_idx]
    vel_dm = vel_dm[sort_idx]

    #find reference particles in this snapshot
    match_idx = np.searchsorted(ids_dm, ids_lmc_ref)
    valid = (match_idx < ids_dm.size) & (ids_dm[match_idx] == ids_lmc_ref)
    coord_lmc = coord_dm[match_idx[valid]]

    # safety check
    #valid = (ids_dm[match_idx] == ids_lmc_ref) #what index in sorted list from not ref snap matches id_ref_sample
    tracked_coords[snap] = coord_dm[match_idx[valid]]

    print("x range:", coord_lmc[:,0].min(), coord_lmc[:,0].max())
    print("y range:", coord_lmc[:,1].min(), coord_lmc[:,1].max())
    print("z range:", coord_lmc[:,2].min(), coord_lmc[:,2].max())

plt.figure()
plt.xlabel('x in kpc')
plt.ylabel('y in kpc')

for snap in snapshots:
    coords = tracked_coords[snap]
    plt.scatter(coords[:,0], coords[:,1],s=2, alpha=0.7, label=f'Snapshot {snap}')
    print(f"Snapshot {snap}")
plt.legend()
plt.title('Traced LMC particles (same ParticleIDs)')
plt.show()
```


# 3: So what? (What does it mean?)
## 3.1 Describe your results
![vR](images/re21.png)

*Figure 1: v(R) vs R for four snapshots*

Figure 1 shows the Circular Velocity (v(R) in km/s) as a function of the Radius (R in kpc) from the center of the galaxy. For Snapshot 105 and 111 the velocity rises sharply and peaks at roughly 1200 km/s near 25 kpc. The curve gently declines. In Snapshot 139 a massive hump appears between 30 and 75 kpc. The velocity peaks much higher, around 1450 km/s, at R $\approx 50$ kpc. This suggests a large amount of mass has entered this specific radial range, likely the LMC reaching a closer point in its orbit. The "hump" shifts further out in Snapshot 153, now peaking near 75 kpc. This indicates the LMC is moving outward again or that the wake it created in the dark matter halo is propagating to larger radii.

![omgR](images/re22.png)

*Figure 2: ω(R) vs R for four snapshots*

Figure 2 shows the Angular Velocity (ω(R) = v(R)/R in km/s/kpc). Unlike the above graph, all four lines almost perfectly overlap. The fact that these lines don't change much between snapshots suggests that while the total velocity (v) changes because of the LMC's mass, the rotational structure of the galaxy's core remains unaffected by the LMC.

These plots are beneficial to the project because while the position scatter plots show where the particles are, the v(R) hump proves those particles are exerting a gravitational effect. This hints to the idea of capturing particle IDs and tracing particles that form the LMC "trail" of dark matter.


![posxyR](images/re23.png)

*Figure 2: Subhalo positions of Y vs X, marked are LMC and MW*

While this graph doesn't directly provide any itnerpretations for tracing data, the code helped us build on the tracers code and find centres.


![trck](images/re24.png)

*Figure 2: Subhalo positions of Y vs X, marked are LMC and MW*

This graph clearly presents the different clumps of data for different snapshots. We assume that the 'outliers' are tidally stripped data. While the code and graph is lakicng, it is just the base which we will work on to find actual LMC traced particles. This graph and code gave us a good understanding of the data, and we have a better plan of action on futher steps.

# 4. Now what? (What's next?)
## 4.1 Plan for the next week

I achieved the intended goal of plotting rotation curves of all four snapshots. We also have a successful code that traces desired data across snapshots. Next week, we will work on finding a way to differentiate and trace LMC only particles from the DM data and calculate the positions from their LMC centres, so we can study the behaviour and effect of those particles on the DM halo.

# 5. Bibliography

[1] Calore et al., *Simulated Milky Way analogues: implications for dark matter indirect searches*, JCAP 12, 053 (2015) [arXiv:1509.02164 [astro-ph.GA]]